# pic0rick RP2350 DSP firmware — step-by-step example

Targets the **`-DDSP`** firmware build (see `docs/dsp_test_guide.md`). Run the
cells top to bottom.

Shows the **raw text replies from the Pico** (parameters + per-stage DSP µs
times) alongside the decoded captures, **arms the pulser from the start**, covers
**raw / Hilbert-envelope / A-law**, runs the DSP **self-test**, checks the
**pulser order**, and **saves every capture to files** (`captures/<name>/`:
`<payload>.npy`, `header.json`, `status.txt`).

Requires: `pyserial`, `numpy`, `matplotlib`.

## 1. Imports

In [ ]:
%matplotlib inline
import os, re, json, dataclasses
import numpy as np
import matplotlib.pyplot as plt
from pic0rick.device import Pic0rick
from pic0rick import dsp

## 2. Connect

`PORT = None` auto-detects the USB-CDC port.

In [ ]:
PORT = None
probe = Pic0rick(port=PORT, verbose=False)

## 3. Version

In [ ]:
print(probe.version()['raw'])

## 4. Status

The Pico's exact `status` line, then a readable breakdown (parameters +
per-stage DSP µs). Only works on a DSP build.

In [ ]:
st = probe.status()
print('Pico reply:')
print(' ', st['raw'])
print()
print(dsp.describe_status(st))

## 5. Commands (`help`)

In [ ]:
def send(cmd, wait=0.4):
    """Send a text command, print and return the Pico's reply. Do NOT use for
    commands that stream binary frames (selftest / captures)."""
    probe.ser.write((cmd + '\n').encode('ascii'))
    reply = ''.join(b.decode('utf-8', 'replace') for b in probe.sread()).strip()
    print(cmd, '->', reply)
    return reply

send('help')

## 6. Set gain and ARM the pulser (from the start)

The pulser is armed here so **every capture below transmits**. The DSP build's
gain command is `dac write <n>` (0..1023).

In [ ]:
GAIN = 50            # TGC DAC value 0..1023
send(f'dac write {GAIN}')
send('dsp scale 512')                       # A-law full-scale reference (ADC counts)
send('pulse config 96 6000 96 neg-first')   # neg/damp/pos ns, order
send('pulser arm')

## 7. Capture helper (saves to files)

`grab(payload, name)` captures one frame, prints the Pico reply + header,
**saves** `captures/<name>/<payload>.npy` + `header.json` + `status.txt`, and
returns `(frame, samples)`.

In [ ]:
CAPT = 'captures'

def grab(payload, name):
    frame = probe.capture(payload)          # 'raw' | 'envelope' | 'alaw'
    s = frame.samples()
    h = frame.header
    print('Pico reply:', probe.last_reply)
    print('  header: samples=%d bytes=%d dc_mean=%.1f envelope_peak=%.1f seq=%d'
          % (h.sample_count, h.payload_bytes, h.adc_dc_mean, h.envelope_peak, h.sequence))
    d = os.path.join(CAPT, name)
    os.makedirs(d, exist_ok=True)
    np.save(os.path.join(d, f'{payload}.npy'), s)
    with open(os.path.join(d, 'header.json'), 'w') as f:
        json.dump(dataclasses.asdict(h), f, indent=2)
    with open(os.path.join(d, 'status.txt'), 'w') as f:
        f.write(dsp.describe_status(probe.status()))
    print('  saved ->', d, '(%s.npy, header.json, status.txt)' % payload)
    return frame, s

## 8. `read_raw` — 8000 raw ADC samples (no FFT)

In [ ]:
_, raw = grab('raw', 'raw')
print('  data: min/mean/max = %d/%d/%d' % (int(raw.min()), int(raw.mean()), int(raw.max())))
plt.figure(figsize=(9, 3)); plt.plot(raw)
plt.title('read_raw (8000 ADC samples)'); plt.xlabel('sample'); plt.ylabel('ADC code'); plt.show()

## 9. `read_fft` — 4096-sample Hilbert envelope

After the capture, the `status` breakdown shows the **per-stage DSP µs** for the
frame that just ran.

In [ ]:
_, env = grab('envelope', 'envelope')
print()
print(dsp.describe_status(probe.status()))
plt.figure(figsize=(9, 3)); plt.plot(env)
plt.title('read_fft (4096-pt Hilbert envelope)'); plt.xlabel('sample'); plt.ylabel('envelope'); plt.show()

## 10. A-law — compressed envelope

`acq alaw` returns the envelope A-law-compressed to `uint8`. Decode it back to
ADC-count amplitude with `dsp.alaw_decode(bytes, reference)` and overlay it on
the float Hilbert envelope from step 9 — they should track closely.

In [ ]:
alaw_frame, alaw = grab('alaw', 'alaw')
reference = alaw_frame.header.alaw_reference
decoded = dsp.alaw_decode(alaw, reference)
print('  alaw uint8 min/max = %d/%d  reference=%.1f' % (int(alaw.min()), int(alaw.max()), reference))
fig, ax = plt.subplots(2, 1, figsize=(9, 5))
ax[0].plot(alaw, color='tab:orange'); ax[0].set_title('A-law bytes (uint8)'); ax[0].set_ylabel('code 0..255')
ax[1].plot(env, label='Hilbert envelope')
ax[1].plot(decoded, '--', label='A-law decoded')
ax[1].set_title('A-law decoded vs Hilbert envelope'); ax[1].set_xlabel('sample'); ax[1].legend()
fig.tight_layout(); plt.show()

## 11. DSP self-test

`dsp selftest` streams deterministic synthetic vectors (7 cases x raw/envelope/
alaw) — a firmware-only check of the DSP pipeline. We read the frames directly
with a `FrameReader`.

In [ ]:
probe.ser.reset_input_buffer()
probe.ser.write(b'dsp selftest\n')
ok = probe.ser.readline().decode('utf-8', 'replace').strip()
print('Pico reply:', ok)
n = int(re.search(r'frames=(\d+)', ok).group(1))
reader = dsp.FrameReader(probe.ser)
frames = [reader.read_frame() for _ in range(n)]
print('read %d self-test frames' % len(frames))
for f in frames:
    if f.header.payload_name == 'envelope':
        print('  case %d %-10s envelope_peak=%.1f'
              % (f.header.selftest_case, dsp.SELFTEST_NAMES[f.header.selftest_case]
                 if f.header.selftest_case < len(dsp.SELFTEST_NAMES) else '?', f.header.envelope_peak))

## 12. Pulser order check

Capture a raw trace with each polarity order and compare the transmit region.

In [ ]:
send('pulse config 96 6000 96 pos-first')
raw_pos = probe.read_raw().samples().astype(int) - 512
send('pulse config 96 6000 96 neg-first')
raw_neg = probe.read_raw().samples().astype(int) - 512
plt.figure(figsize=(9, 3))
plt.plot(raw_pos[:80], label='pos-first')
plt.plot(raw_neg[:80], label='neg-first')
plt.title('pulser order — transmit region (first 80 samples)')
plt.xlabel('sample'); plt.ylabel('ADC - 512'); plt.legend(); plt.show()

## 13. Saved files

In [ ]:
for root, _, files in sorted(os.walk(CAPT)):
    for fn in sorted(files):
        p = os.path.join(root, fn)
        print('%8d  %s' % (os.path.getsize(p), p))

## 14. Disarm the pulser

In [ ]:
send('pulser disarm')

---
See `docs/dsp_test_guide.md` for the full command set, `pic0rick.dsp` for the
protocol, and `python/example_dsp.py` for a script version. A-law needs
`dsp.SELFTEST_NAMES` / `alaw_decode` from `pic0rick.dsp`.